In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

# Dash + UI components
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output
import plotly.express as px

# Utilities
import base64
import pandas as pd

# CRUD Module (Model access)
from animalshelter import AnimalShelter



# Data Manipulation / Model

username = "aacuser"
password = "SNHU1234"

db = AnimalShelter(username, password)


def _df_from_docs(docs):
    """Convert Mongo docs -> clean DataFrame (drop _id if present)."""
    dff = pd.DataFrame.from_records(docs)
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)
    return dff


# Initial load (unfiltered)
df = _df_from_docs(db.read({}))



# Queries (per Spec Document)

WATER_Q = {
    "animal_type": "Dog",
    "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}

MOUNTAIN_Q = {
    "animal_type": "Dog",
    "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
}

DISASTER_Q = {
    "animal_type": "Dog",
    "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
}


def query_for(filter_type: str) -> dict:
    if filter_type == "water":
        return WATER_Q
    if filter_type == "mountain":
        return MOUNTAIN_Q
    if filter_type == "disaster":
        return DISASTER_Q
    return {}  



# Dashboard Layout / View

app = JupyterDash(__name__)

# Logo (put Grazioso Salvare logo file in same folder as notebook)
# If your logo file name is different, change it here:
image_filename = "Grazioso Salvare Logo.png"

encoded_image = None
try:
    encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()
except Exception:
    encoded_image = None

logo_block = (
    html.Img(
        src=f"data:image/png;base64,{encoded_image}",
        style={"height": "110px", "margin": "10px"},
    )
    if encoded_image
    else html.Div(
        "Logo file not found. Put GraziosoSalvareLogo.png next to the notebook (or update image_filename).",
        style={"color": "crimson", "margin": "10px"},
    )
)

app.layout = html.Div(
    [
        html.Center(html.B(html.H1("SNHU CS-340 Dashboard"))),
        html.Center(html.B("Unique ID: Project Two - Tyler Hubbell")),
        html.Hr(),

        # Logo
        html.Center(logo_block),
        html.Hr(),

        # Filter Options (Controller)
        html.Div(
            [
                html.H3("Rescue Type Filter"),
                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {"label": "Reset (All)", "value": "reset"},
                        {"label": "Water Rescue", "value": "water"},
                        {"label": "Mountain/Wilderness Rescue", "value": "mountain"},
                        {"label": "Disaster/Tracking", "value": "disaster"},
                    ],
                    value="reset",   # IMPORTANT: default unfiltered state
                    inline=True,
                ),
            ],
            style={"margin": "10px"},
        ),

        html.Hr(),

        # Data Table
        dash_table.DataTable(
            id="datatable-id",
            columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
            data=df.to_dict("records"),
            page_size=10,
            sort_action="native",
            filter_action="native",
            row_selectable="single",
            selected_rows=[0],
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "minWidth": "120px", "maxWidth": "250px", "whiteSpace": "normal"},
        ),

        html.Br(),
        html.Hr(),

        # Charts side-by-side
        html.Div(
            className="row",
            style={"display": "flex"},
            children=[
                html.Div(
                    id="graph-id",
                    className="col s12 m6",
                    style={
                    "flex": "1",
                    "padding": "10px",
                    "minWidth": "520px",
                    "overflowX": "auto"
                    },
                ),

                html.Div(id="map-id", 
                         className="col s12 m6", 
                         style={"flex": "1", "padding": "10px"}),
            ],
        ),
    ]
)



# Interaction Between Components / Controller

@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value"),
)
def update_dashboard(filter_type):
    q = query_for(filter_type)
    dff = _df_from_docs(db.read(q))
    return dff.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
)
def update_graphs(viewData):
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or "breed" not in dff.columns:
        return [html.Div("No data to display for this filter.")]

    # Keep only the top 10 breeds (everything else grouped into "Other")
    vc = dff["breed"].fillna("Unknown").value_counts()
    top10 = vc.head(10)

    dff2 = dff.copy()
    dff2["breed"] = dff2["breed"].fillna("Unknown")
    dff2.loc[~dff2["breed"].isin(top10.index), "breed"] = "Other"

    fig = px.pie(dff2, names="breed", title="Breed Distribution (Top 10 + Other)")
    fig.update_traces(
        textinfo="label+percent",   # displays breed and percentage on the chart
        textposition="inside"
    )
    fig.update_layout(
        showlegend=False, 
        margin={"l": 10, "r": 10, "t": 50, "b": 10},
    )

    return [dcc.Graph(figure=fig)]




@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns"),
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [
        {
            "if": {"column_id": i},
            "background_color": "#D2F3FF",
        }
        for i in selected_columns
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(viewData, index):
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return [html.Div("No map data available.")]

    # default selected row
    row = 0
    if index and len(index) > 0:
        row = index[0]

    # Ensure columns exist
    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return [html.Div("Missing location_lat / location_long columns in data.")]

    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]

    # fallback if lat/lon missing for that row
    if pd.isna(lat) or pd.isna(lon):
        lat, lon = 30.75, -97.48

    breed = str(dff.loc[row, "breed"]) if "breed" in dff.columns else "Unknown"
    name = str(dff.loc[row, "name"]) if "name" in dff.columns else ""

    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([html.H1("Animal Name"), html.P(name)]),
                    ],
                ),
            ],
        )
    ]


# Run app and display result in JupyterLab mode
app.run_server()


Dash app running on https://vaticanpasta-caesarbeauty-3000.codio.io/proxy/8050/
